# MDR-TS v18.3
**Temporal-Station Baseline Model for Soil Moisture Prediction**

**Author:** Jakob Balkovec  
**Affiliation:** Seattle University, Computer Science  
**Project:** MDR
**Notebook Type:** Training & Evaluation  
**Last Updated:** Tue Jan 6th 2026

---

## Model Summary
- **Model Name:** MDR-TS  
- **Version:** v18.3
- **Task:** Regression (Soil Moisture at 5 cm depth)  
- **Target Variable:** `soil_moisture_5cm`  
- **Temporal Resolution:** Daily  

---

## Reproducibility
- **Random Seed:** 42
- **Split Metadata:** `data/splits/derived_8.0/split_meta.json`
- **Environment:** Google Colab / VS Code Remote Kernel

---

Adapted for a Macbook M2 Pro environment.

**What's new?**

Calibration

## 0. Imports

In [42]:
import os
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import GroupKFold

import optuna

from xgboost import XGBRegressor

import torch

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("imports loaded")
print(f"using: {device}")

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

imports loaded
using: cpu


In [29]:
SEED = 42
DEEP_SEARCH = 40

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Random seed set to {SEED}")

def print_env_info():
    print("Environment information:")
    print(f"  Python version: {os.sys.version.split()[0]}")
    print(f"  NumPy version:  {np.__version__}")
    print(f"  Pandas version: {pd.__version__}")

    try:
        import xgboost
        print(f"  XGBoost version: {xgboost.__version__}")
    except ImportError:
        print("  XGBoost not installed")

    IN_COLAB = "COLAB_GPU" in os.environ
    print(f"  Running in Colab: {IN_COLAB}")

    if IN_COLAB:
        gpu = os.environ.get("COLAB_GPU", None)
        print(f"  GPU available: {gpu}")
    else:
        print("  GPU available: False")

print_env_info()

plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("environment setup complete")

Random seed set to 42
Environment information:
  Python version: 3.10.18
  NumPy version:  1.26.4
  Pandas version: 2.0.3
  XGBoost version: 3.2.0
  Running in Colab: False
  GPU available: False
environment setup complete


In [30]:
VERSION = "v18"
SUBVERSION = "v18.3"
RUN_NAME = "mdr_ts_v18_3"

PROJECT_ROOT = "/Users/jbalkovec/Desktop/MDR"
DATA_ROOT = f"{PROJECT_ROOT}/Temporal/Pipeline/data"
SPLIT_ROOT = f"{DATA_ROOT}/splits"
OUTPUT_ROOT = f"{PROJECT_ROOT}/Models/Temporal/{VERSION}/{SUBVERSION}"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Project paths:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  DATA_ROOT:    {DATA_ROOT}")
print(f"  SPLIT_ROOT:   {SPLIT_ROOT}")
print(f"  OUTPUT_ROOT:  {OUTPUT_ROOT}")

print("\nKey file checks:")
print("  data exists:",
      os.path.exists(DATA_ROOT))
print("  splits exists:",
      os.path.exists(SPLIT_ROOT))
print("  output exists:",
      os.path.exists(OUTPUT_ROOT))

Project paths:
  PROJECT_ROOT: /Users/jbalkovec/Desktop/MDR
  DATA_ROOT:    /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data
  SPLIT_ROOT:   /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits
  OUTPUT_ROOT:  /Users/jbalkovec/Desktop/MDR/Models/Temporal/v18/v18.3

Key file checks:
  data exists: True
  splits exists: True
  output exists: True


In [31]:
TRAIN_PATH = str(Path(SPLIT_ROOT) / "derived_8.0/train.csv")
VAL_PATH   = str(Path(SPLIT_ROOT) / "derived_8.0/val.csv")
TEST_PATH  = str(Path(SPLIT_ROOT) / "derived_8.0/test.csv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing split file: {p}")

print("Split files:")
print(" ", TRAIN_PATH)
print(" ", VAL_PATH)
print(" ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

Split files:
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_8.0/train.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_8.0/val.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_8.0/test.csv


In [32]:
print("Common columns across splits:",
      len(set(train_df.columns) & set(val_df.columns) & set(test_df.columns)))

print("\ncolumns:")
print(list(train_df.columns)[:30])

Common columns across splits: 495

columns:
['station_id', 'date', 'longitude', 'latitude', 'precip_mm', 's1_vv', 's1_vh', 's2_b4', 's2_b8', 's2_b11', 's2_b12', 'LST_modis', 'elev', 'slope', 'aspect', 'DOY', 'SMAP_sm_am_interp', 'SMAP_sm_pm_interp', 'soil_moisture_5cm', 'J_aspect_deg', 'J_bio_bio01', 'J_bio_bio02', 'J_bio_bio03', 'J_bio_bio04', 'J_bio_bio05', 'J_bio_bio06', 'J_bio_bio07', 'J_bio_bio08', 'J_bio_bio09', 'J_bio_bio10']


In [33]:
TARGET_COL = "soil_moisture_5cm"

KEEP_META_COLS = ["station_id", "date", "longitude", "latitude"]

FEATURE_COLS = [
        "SMAP_sm_pm_interp_ema02",
        "V_rollmin_LST_modis_kobs30",
        "D_sin_DOY",
        "G_rain_sum_3d",
        "V_ema_G_API_kobs7",
        "V_rollmin_G_API_kobs30",
        "G_rain_sum_7d",
        "C_lag_LST_modis_kobs30",
        "C_lag_G_API_kobs1",
        "V_ema_G_API_kobs14",
        "V_rollmean_G_API_kobs14",
        "G_API",
        "A_pct_G_API",
        "V_rollcv_G_API_kobs30",
        "G_DSLR",
        "SMAP_ampm_diff_interp",
        "V_rollmax_G_API_kobs30",
        "V_rollmin_G_API_kobs7",
        "V_ema_G_API_kobs30",
        "V_rollmean_s2_b11_kobs7",
        "V_ema_LST_modis_kobs7",
        "C_smm_G_API_alpha0.85_n5",
        "C_lag_G_API_kobs5",
        "V_rollmean_G_API_kobs7",
        "C_lag_s2_b11_kobs30",
        "D_z_LST_modis",
        "A_d_G_API_kobs1",
        "V_rollcv_LST_modis_kobs30",
        "V_rollcv_G_API_kobs7",
        "V_rollstd_LST_modis_kobs30",
        "A_d_E_SAR_diff_kobs14",
        "C_lag_G_API_kobs6",
        "V_rollrng_F_NDMI_kobs7",
        "V_rollcv_G_API_kobs14",
        "C_lag_LST_modis_kobs6",
        "A_d_E_SAR_diff_kobs30",
        "A_d_LST_modis_kobs14",
        "SMAP_sm_am_interp_rollrange7",
        "V_rollstd_LST_modis_kobs14",
        "D_fft_ent_E_SAR_ratio_kobs30",
        "A_d_E_SAR_diff_kobs5",
        "SMAP_sm_pm_interp_rollrange7",
        "V_rollstd_F_NDMI_kobs7",
        "V_rollstd_E_SAR_ratio_kobs7",
        "V_rollrng_E_SAR_diff_kobs7",
        "V_rollstd_s2_b12_kobs7",
        "A_grad_E_SAR_diff_kobs14",
        "D_fft_dom_LST_modis_kobs30",
        "V_rollcv_s2_b12_kobs7",
        "A_d_E_SAR_ratio_kobs5",
        "D_fft_ent_LST_modis_kobs30",
        "V_rollstd_F_NDVI_kobs7",
        "A_grad_s2_b12_kobs7",
        "A_pct_F_NDVI",
        "A_d_s2_b12_kobs2",
        "A_grad_E_SAR_diff_kobs30",
        "A_d_F_NDVI_kobs2",
        "A_grad_E_SAR_diff_kobs7",
        "SMAP_sm_interp_rollrange7",
        "A_d_s2_b12_kobs7",
        "A_d_F_NDVI_kobs1",
        "V_rollcv_LST_modis_kobs14",
        "SMAP_sm_am_interp_rollstd7",
        "V_rollstd_SMAP_sm_interp_kobs7",
        "A_d_s2_b12_kobs5",
        "A_pct_SMAP_sm_interp",
        "SMAP_sm_am_interp_pctchg",
        "V_rollrng_SMAP_sm_interp_kobs7",
        "SMAP_sm_interp_pctchg",
        "A_d_E_SAR_diff_kobs2",
        "G_DSLR_isnan",
        "SMAP_sm_pm_interp_mask",
        "SMAP_sm_interp_mask",
        "SMAP_sm_am_interp_mask",
        "SMAP_sm_am_interp_diff1",
        "SMAP_sm_interp_rollstd7",
        "SMAP_sm_interp_diff1",
        "V_rollstd_E_SAR_diff_kobs7",
        "A_grad_s2_b12_kobs14",
        "A_d_E_SAR_ratio_kobs7",
        "A_grad_LST_modis_kobs14",
        "A_d_SMAP_sm_interp_kobs1",
        "SMAP_sm_pm_interp_pctchg",
        "A_grad_E_SAR_ratio_kobs7",
        "A_d_E_SAR_diff_kobs1",
        "A_d_E_SAR_diff_kobs7",
        "SMAP_sm_pm_interp_diff1",
        "A_d_F_NDVI_kobs5",
        "A_d_E_SAR_ratio_kobs2",
        "A_d_G_API_kobs5",
        "A_d_SMAP_sm_interp_kobs2",
        "D_fft_dom_E_SAR_ratio_kobs30",
        "SMAP_sm_pm_interp_rollstd7",
        "V_rollrng_s2_b12_kobs7",
        "V_rollrng_F_NDVI_kobs7",
        "A_d_SMAP_sm_interp_kobs14",
        "A_pct_E_SAR_ratio",
        "V_rollstd_SMAP_sm_interp_kobs30",
        "A_d_E_SAR_ratio_kobs1",
        "A_pct_LST_modis",
        "A_grad_SMAP_sm_interp_kobs14",
        "A_pct_E_SAR_diff",
        "SMAP_sm_interp_grad7",
        "A_grad_SMAP_sm_interp_kobs7",
        "A_d_LST_modis_kobs1",
        "V_rollcv_E_SAR_diff_kobs7",
        "A_d_s2_b11_kobs5",
        "V_rollstd_LST_modis_kobs7",

        # drift
        'year_frac',
        'sin_year',
        'cos_year',
        'API_x_year',
        'SMAP_x_year',


        # spatial
        "slope",
        "elev",
        "K_slope_sin",
        "K_slope_cos",
        "K_aspect_cos",

        "J_clay_wfrac_b0",
        "J_sand_wfrac_b0",
        "J_sand_clay_ratio_b0",

]

expected = set(KEEP_META_COLS + FEATURE_COLS + [TARGET_COL])
missing_train = sorted(list(expected - set(train_df.columns)))
missing_val   = sorted(list(expected - set(val_df.columns)))
missing_test  = sorted(list(expected - set(test_df.columns)))

if missing_train or missing_val or missing_test:
    raise ValueError(
        f"Missing columns:\n"
        f"  train: {missing_train}\n"
        f"  val:   {missing_val}\n"
        f"  test:  {missing_test}"
    )

print("Columns locked")
print("  Features:", len(FEATURE_COLS))
print("  Target:  ", TARGET_COL)

Columns locked
  Features: 121
  Target:   soil_moisture_5cm


In [34]:
def metrics_block(y_true, y_pred, name="Test", tol=None, prefix=""):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    err = y_true - y_pred
    ae = np.abs(err)

    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)

    bias = float(np.mean(err))
    err_std = float(np.std(err, ddof=0))
    debiased_std = float(np.std(err - bias, ddof=0))

    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        corr = np.nan
    else:
        corr = float(np.corrcoef(y_true, y_pred)[0, 1])

    q_err = np.quantile(err, [0.05, 0.25, 0.50, 0.75, 0.95])
    med_ae = float(np.median(ae))
    p90_ae = float(np.quantile(ae, 0.90))

    print(f"{name}")
    print(f"{prefix}n    = {len(y_true)}")
    print(f"{prefix}R2   = {float(r2):.5f}")
    print(f"{prefix}MAE  = {float(mae):.5f}")
    print(f"{prefix}RMSE = {float(rmse):.5f}")
    print(f"{prefix}Bias (true-pred)  = {bias:.5f}")
    print(f"{prefix}Err std           = {err_std:.5f}")
    print(f"{prefix}Err std (debiased)= {debiased_std:.5f}")
    print(f"{prefix}Pearson r         = {corr:.5f}")
    print(f"{prefix}Median |err|      = {med_ae:.5f}")
    print(f"{prefix}P90 |err|         = {p90_ae:.5f}")
    print(f"{prefix}err quantiles [5/25/50/75/95%] = "
          f"{q_err[0]:.5f}, {q_err[1]:.5f}, {q_err[2]:.5f}, {q_err[3]:.5f}, {q_err[4]:.5f}")

    if tol is not None:
        tols = tol if isinstance(tol, (list, tuple, np.ndarray)) else [tol]
        for t in tols:
            frac = float(np.mean(ae <= t))
            print(f"{prefix}Within ±{t}: {frac:.3%}")

    return {
        "n": int(len(y_true)),
        "r2": float(r2),
        "mae": float(mae),
        "rmse": float(rmse),
        "bias_true_minus_pred": float(bias),
        "err_std": float(err_std),
        "err_std_debiased": float(debiased_std),
        "pearson_r": float(corr),
        "median_abs_err": float(med_ae),
        "p90_abs_err": float(p90_ae),
        "err_q05": float(q_err[0]),
        "err_q25": float(q_err[1]),
        "err_q50": float(q_err[2]),
        "err_q75": float(q_err[3]),
        "err_q95": float(q_err[4]),
    }

### Split Strategy
- **Training set:**  
  Two stations, early time period  
- **Validation set:**  
  Same stations as training, held-out **future dates** (temporal holdout)
- **Test set:**  
  One completely unseen station (station-level holdout)

### Motivation
- Validation evaluates **temporal generalization** on known stations
- Test evaluates **spatial generalization** to an unseen station
- This avoids spatial leakage while preserving sufficient training data

In [35]:
print("=== SPLIT SUMMARY ===")

def split_summary(name, d):
    print(f"\n{name.upper()}")
    print(f"  rows:     {len(d)}")
    print(f"  stations: {sorted(d['station_id'].unique().tolist())}")
    if "date" in d.columns:
        print(f"  date range: {d['date'].min()} -- {d['date'].max()}")

split_summary("train", train_df)
split_summary("val", val_df)
split_summary("test", test_df)

print("\n=== LEAKAGE CHECK ===")
print("train ∩ test:", sorted(set(train_df.station_id) & set(test_df.station_id)))
print("val   ∩ test:", sorted(set(val_df.station_id) & set(test_df.station_id)))

print("\n-- split locked --")


=== SPLIT SUMMARY ===

TRAIN
  rows:     6868
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824']
  date range: 2017-01-01 -- 2020-12-31

VAL
  rows:     2720
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
  date range: 2021-01-01 -- 2022-12-31

TEST
  rows:     4016
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824']
  date range: 2023-01-01 -- 2025-12-31

=== LEAKAGE CHECK ===
train ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824']
val   ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']

-- split locked --


In [36]:
trainval_df_d = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

trainval_df_d["date"] = pd.to_datetime(trainval_df_d["date"], errors="coerce")
trainval_df_d["year"] = trainval_df_d["date"].dt.year.astype(float)

max_year = trainval_df_d["year"].max()
beta = 0.2

w_trainval = np.exp(beta * (trainval_df_d["year"] - max_year))
w_trainval = w_trainval / w_trainval.mean()

In [37]:
trainval_df_d = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

X_trainval_d = trainval_df_d[FEATURE_COLS].copy()
y_trainval_d = trainval_df_d[TARGET_COL].copy()

X_test_d = test_df[FEATURE_COLS].copy()
y_test_d = test_df[TARGET_COL].copy()

print("\nDRIFT matrices:")
print("  X_trainval_d:", X_trainval_d.shape)
print("  X_test_d:    ", X_test_d.shape)


DRIFT matrices:
  X_trainval_d: (9588, 121)
  X_test_d:     (4016, 121)


### Drift Model (No Weights)

In [47]:
def objective(trial):

    params = {
        "objective": "reg:absoluteerror",
        "random_state": SEED,
        "n_jobs": -1,

        "subsample": trial.suggest_float("subsample", 0.85, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 0.9),

        "max_depth": trial.suggest_int("max_depth", 8, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 3),

        "n_estimators": trial.suggest_int("n_estimators", 5500, 8500),
        "learning_rate": trial.suggest_float("learning_rate", 0.025, 0.045),

        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 2.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 0.05),

        "gamma": trial.suggest_float("gamma", 0.0, 0.2),
    }

    X = np.asarray(X_trainval_d)
    y = np.asarray(y_trainval_d).ravel()
    groups = np.asarray(trainval_df_d["year"]).ravel()

    gkf = GroupKFold(n_splits=5)
    scores = []

    for train_idx, val_idx in gkf.split(X, y, groups=groups):
        model = XGBRegressor(**params)

        model.fit(
            X[train_idx],
            y[train_idx],
            verbose=0
        )

        preds = model.predict(X[val_idx])
        scores.append(r2_score(y[val_idx], preds))

    return float(np.mean(scores))

In [48]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, timeout=1800)

[I 2026-02-23 15:59:24,253] A new study created in memory with name: no-name-0fd335b5-ad26-458a-ad8f-83dea67bcd85
[I 2026-02-23 16:08:10,079] Trial 0 finished with value: 0.7981070706691848 and parameters: {'subsample': 0.9693179658758008, 'colsample_bytree': 0.8959623541806981, 'max_depth': 10, 'min_child_weight': 3, 'n_estimators': 7434, 'learning_rate': 0.03867633801523241, 'reg_lambda': 1.1633268294420005, 'reg_alpha': 0.02715347831000415, 'gamma': 0.11087475551887796}. Best is trial 0 with value: 0.7981070706691848.
[W 2026-02-23 16:11:20,138] Trial 1 failed with parameters: {'subsample': 0.9843400453628861, 'colsample_bytree': 0.7626846638463849, 'max_depth': 9, 'min_child_weight': 1, 'n_estimators': 7815, 'learning_rate': 0.03890084116275222, 'reg_lambda': 1.785204198720458, 'reg_alpha': 0.02137227202093301, 'gamma': 0.10754857633126723} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/jbalkovec/miniforge3/envs/dr-arm/lib/pyt

KeyboardInterrupt: 

In [ ]:
best_params = study.best_params

best_params.update({
    "objective": "reg:absoluteerror",
    "random_state": SEED,
    "n_jobs": -1,
})

final_model = XGBRegressor(**best_params)

trainval_df = pd.concat([train_df, val_df], axis=0)

final_model.fit(
    trainval_df[FEATURE_COLS],
    trainval_df[TARGET_COL],
    verbose=0,
)

test_pred = final_model.predict(test_df[FEATURE_COLS])

_ = metrics_block(
    y_test_d,
    test_pred,
    name="=== DRIFT (no WEIGHTS) Test (OPTUNA) ===",
    tol=[0.02, 0.05, 0.10]
)

=== DRIFT (no WEIGHTS) Test ===
n    = 4016
R2   = 0.82186
MAE  = 0.02925
RMSE = 0.03974
Bias (true-pred)  = -0.00218
Err std           = 0.03968
Err std (debiased)= 0.03968
Pearson r         = 0.90686
Median |err|      = 0.02246
P90 |err|         = 0.06176
err quantiles [5/25/50/75/95%] = -0.06786, -0.02352, -0.00259, 0.02141, 0.05540
Within ±0.02: 44.821%
Within ±0.05: 84.537%
Within ±0.1: 97.012%


### Best Hyperparameters
```yaml
  unweighted_best_r2:
    objective: reg:absoluteerror
    random_state: ${SEED}
    n_jobs: -1
    subsample: 0.9
    colsample_bytree: 0.8
    max_depth: 9
    min_child_weight: 1
    n_estimators: 5500
    learning_rate: 0.04
    reg_lambda: 1.5
    reg_alpha: 0.03
    gamma: 0.0
```

### Best Hyperparameters (Optuna Tuned)
```yaml

```

### Drift Model (With Weights)

In [ ]:
w_trainval_s = pd.Series(w_trainval)
years_tv = trainval_df["year"].reset_index(drop=True)

print("Weight stats:")
print(w_trainval_s.describe())

min_year = years_tv.min()
max_year = years_tv.max()

print("Min year weight:", float(w_trainval_s[years_tv == min_year].mean()))
print("Max year weight:", float(w_trainval_s[years_tv == max_year].mean()))

In [ ]:
trainval_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

X_tv = trainval_df[FEATURE_COLS].to_numpy()
y_tv = trainval_df[TARGET_COL].to_numpy().ravel()
w_tv = np.asarray(w_trainval).ravel()

assert len(X_tv) == len(y_tv) == len(w_tv), "X/y/weights length mismatch"

In [ ]:
def objective_weighted(trial):

    params = {
        "objective": "reg:pseudohubererror",
        "random_state": SEED,
        "n_jobs": -1,

        "subsample": trial.suggest_float("subsample", 0.85, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 0.9),

        "max_depth": trial.suggest_int("max_depth", 7, 9),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 4),

        "n_estimators": trial.suggest_int("n_estimators", 5500, 9000),
        "learning_rate": trial.suggest_float("learning_rate", 0.025, 0.05),

        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 2.2),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 0.06),

        "gamma": trial.suggest_float("gamma", 0.0, 0.25),
    }

    gkf = GroupKFold(n_splits=5)
    scores = []

    groups = trainval_df["year"].values

    for train_idx, val_idx in gkf.split(X_tv, y_tv, groups=groups):

        model = XGBRegressor(**params)

        model.fit(
            X_tv[train_idx],
            y_tv[train_idx],
            sample_weight=w_tv[train_idx],
            verbose=0
        )

        preds = model.predict(X_tv[val_idx])
        scores.append(r2_score(y_tv[val_idx], preds))

    return float(np.mean(scores))

In [ ]:
study_w = optuna.create_study(direction="maximize")
study_w.optimize(objective_weighted, timeout=1800)  # 30 minutes

print("Best CV R2:", study_w.best_value)
print("Best params:", study_w.best_params)

In [ ]:
best_params_w = study_w.best_params.copy()
best_params_w.update({
    "objective": "reg:pseudohubererror",
    "random_state": SEED,
    "n_jobs": -1,
})

final_model_w = XGBRegressor(**best_params_w)

final_model_w.fit(
    X_tv, y_tv,
    sample_weight=w_tv,
    verbose=0
)

y_test = np.asarray(test_df[TARGET_COL]).ravel()
pred_test_w = np.asarray(final_model_w.predict(test_df[FEATURE_COLS])).ravel()

_ = metrics_block(
    y_test, pred_test_w,
    name="=== DRIFT (WEIGHTED) Test (OPTUNA) ===",
    tol=[0.02, 0.05, 0.10]
)

=== DRIFT (WEIGHTED) Test ===
n    = 4016
R2   = 0.82503
MAE  = 0.02804
RMSE = 0.03939
Bias (true-pred)  = -0.00308
Err std           = 0.03927
Err std (debiased)= 0.03927
Pearson r         = 0.90892
Median |err|      = 0.01996
P90 |err|         = 0.06096
err quantiles [5/25/50/75/95%] = -0.06623, -0.02264, -0.00329, 0.01749, 0.05603
Within ±0.02: 50.075%
Within ±0.05: 85.458%
Within ±0.1: 96.589%


### Best Hyperparameters (Manually Tuned)
```yaml
  weighted_best_r2:
    objective: reg:absoluteerror
    random_state: ${SEED}
    n_jobs: -1
    subsample: 0.9
    colsample_bytree: 0.8
    max_depth: 8
    min_child_weight: 2
    n_estimators: 5500
    learning_rate: 0.04
    reg_lambda: 1.5
    reg_alpha: 0.03
    gamma: 0.0
```

### Best Hyperparameters (Optuna Tuned)
```yaml

```

---

_Jakob Balkovec_